# Swan — Test profiles on Lina's final catalogue

Same questionnaire, rules and 31 test profiles as `loredana_questionnaire_v2_mvp.ipynb`, but the products now come from Lina's reviewed catalogue.

## Data
| Source | File |
|---|---|
| Lina's reviewed catalogue (152 products, 04/11) | `raw_data/moisturizer_catalogue_keep_reviewed_v1.csv` |

What changes compared to v2:
- no ingredient splitting here: the file already has `inci_text` (clean INCI names, label order)
- products we can recommend = France + at least 5 ingredients
- texture recalculated with my rule (the file still has the old one)

## Settings

In [ ]:
import pandas as pd

INPUT_PATH = "../raw_data/moisturizer_catalogue_keep_reviewed_v1.csv"

SCOPE_COLUMN = "france"
NB_ROUTINE_PRODUCT = 3             # moisturizers shown per profile
BLEMISH_REFERRAL_LEVEL = 4         # Q3: from this level → referral (to validate)
MIN_INGREDIENTS = 5
AVOID_FRAGRANCE_INCLUDES_ALLERGENS = False  # Q7 'avoid fragrance' = only 'parfum' on the label
LIGHT_SKIN_TYPES = ["oily", "combination", "normal", "not_sure"]

# Word lists for the signals — names from EU Regulation 1223/2009, Annex III (fragrance allergens)
FRAGRANCE_ALLERGENS = ["linalool", "limonene", "citronellol", "geraniol", "citral", "coumarin", "eugenol",
                       "benzyl alcohol", "benzyl benzoate", "benzyl salicylate", "hexyl cinnamal", "farnesol",
                       "alpha-isomethyl ionone", "hydroxycitronellal", "cinnamal", "amyl cinnamal", "isoeugenol"]
ESSENTIAL_OILS = ["lavandula", "citrus", "mentha", "eucalyptus", "rosmarinus", "melaleuca", "pelargonium",
                  "cymbopogon", "cananga", "santalum", "pogostemon", "juniperus", "cinnamomum", "jasminum"]
ACNE_ACTIVES = ["salicylic acid", "niacinamide", "zinc pca", "zinc gluconate", "azelaic acid"]
# Texture: heavy ingredients = butters, waxes, oils, petrolatum (INCI order = amount above 1 %)
HEAVY_INGREDIENTS = ["petrolatum", "paraffinum liquidum", "paraffin", "mineral oil", "butter", "cera", "wax",
                     "lanolin", "squalane", "seed oil", "kernel oil", "fruit oil"]
RICH_NAME_WORDS = ["riche", "rich", "baume", "balm", "cold cream", "nourri"]
LIGHT_NAME_WORDS = ["gel", "fluide", "fluid", "matifiant", "light", "légère"]

## Load Lina's catalogue

In [ ]:
moist = pd.read_csv(INPUT_PATH, dtype={"code": str})
print(moist.shape)
print("France:", (moist["france"] == True).sum(), "| EU:", (moist["eu"] == True).sum())
print("Codes unique:", moist["code"].is_unique)
moist[["code", "product_name", "brands", "inci_text"]].head()

In [ ]:
# products we can recommend: France + a complete ingredient list
moist["in_scope"] = (moist[SCOPE_COLUMN] == True) & (moist["nb_ingredients"] >= MIN_INGREDIENTS)
print("In scope:", moist["in_scope"].sum())

# French products removed because the list is broken (they would look fragrance-free)
moist[(moist[SCOPE_COLUMN] == True) & (moist["nb_ingredients"] < MIN_INGREDIENTS)][["code", "product_name", "nb_ingredients"]]

## Signals from the name

In [ ]:
#Check the exclusion based on product name for MVP
names = moist["product_name"].str.lower().fillna("")
moist["is_anti_ageing"] = names.str.contains("anti-âge|anti-age|rides|wrinkle|firming|lift")
moist["is_baby"] = names.str.contains("bébé|bebe|baby")
moist["is_face_and_body"] = names.str.contains("corps|body")
moist[["is_anti_ageing", "is_baby", "is_face_and_body"]].sum()

#Check duplicates: same name + brand = possible duplicate, NOT excluded
# (different codes can be different sizes or formulas; Lina checks them by hand)
moist["name_brand"] = names + " | " + moist["brands"].str.lower().fillna("")
moist["possible_duplicate"] = moist["name_brand"].duplicated(keep=False)
print("Products with a possible duplicate:", moist["possible_duplicate"].sum())

## Signals from the ingredients (same rules as v2)

In [ ]:
moist["has_parfum"] = moist["inci_text"].str.contains("parfum")
moist["has_fragrance_allergen"] = moist["inci_text"].str.contains("|".join(FRAGRANCE_ALLERGENS))
moist["has_essential_oil"] = moist["inci_text"].str.contains("|".join(ESSENTIAL_OILS))
moist["has_acne_active"] = moist["inci_text"].str.contains("|".join(ACNE_ACTIVES))

#review the texture details
def count_heavy(ingredients):
    """How many ingredients of the list are heavy and can identify a rich texture?"""
    count = 0
    for ingredient in ingredients:
        for heavy in HEAVY_INGREDIENTS:
            if heavy in ingredient:
                count = count +1
                break
    return count


def find_texture(inci_text, name):
    """Identify the texture rich, light or medium from the product name and the first ingredients"""

    name = str(name).lower()
    for word in RICH_NAME_WORDS:
        if word in name:
            return "rich"
    for word in LIGHT_NAME_WORDS:
        if word in name and "baume" not in name and "gelée" not in name:
            return "light"

    #if not look at the first ingredients:
    ingredients = inci_text.split(" | ")
    heavy_top3 = count_heavy(ingredients[:3])
    heavy_top5 = count_heavy(ingredients[:5])

    if ingredients[0] != "aqua" or heavy_top3 >= 1 or heavy_top5 >= 2:
        return "rich"
    elif heavy_top5 == 0:
        return "light"
    else:
        return "medium"

textures = [find_texture(text, name) for text, name in zip(moist["inci_text"], moist["product_name"])]
moist["is_rich"] = [texture == "rich" for texture in textures]
moist["is_light"] = [texture == "light" for texture in textures]

signals = ["has_parfum", "has_fragrance_allergen", "has_essential_oil", "has_acne_active", "is_rich", "is_light",
           "is_anti_ageing", "is_baby", "is_face_and_body"]
print("In scope:", moist["in_scope"].sum(), "| light:", moist[moist["in_scope"]]["is_light"].sum(),
      "| rich:", moist[moist["in_scope"]]["is_rich"].sum(), "| without parfum:", (~moist[moist["in_scope"]]["has_parfum"]).sum())
(moist[moist["in_scope"]][signals].mean() * 100).round(1)

## The questionnaire 

In [ ]:
QUESTIONNAIRE = {
    "is_adult": [True, False],
    "concern": ["acne_prone", "other"],
    "blemish_level": [1, 2, 3, 4, 5],
    "referral_flags": [[], ["prescribed_treatment"], ["unusual_lesion"], ["pregnant_or_breastfeeding"]],
    "skin_type": ["oily", "combination", "normal", "dry", "not_sure"],
    "reactivity": [True, False],
    "avoid": [[], ["fragrance"], ["essential_oils"], ["fragrance", "essential_oils"]],
}

def check_profile(profile):
    """List the problems of a profile (empty list = valid)."""
    problems = []
    for field in QUESTIONNAIRE:
        if field not in profile:
            problems.append("missing " + field)
        elif type(profile[field]) == list:                  # Q4 and Q7: several answers possible
            allowed = []
            for option in QUESTIONNAIRE[field]:
                allowed = allowed + option
            for item in profile[field]:
                if item not in allowed:
                    problems.append("wrong value for " + field + ": " + item)
        elif profile[field] not in QUESTIONNAIRE[field]:
            problems.append("wrong value for " + field)
    return problems

print(check_profile({"is_adult": True, "concern": "acne_prone", "blemish_level": 2, "referral_flags": [],
                     "skin_type": "oily", "reactivity": True, "avoid": []}))
print(check_profile({"is_adult": True, "concern": "acne", "skin_type": "very oily"}))

**Data contract** — what each answer needs from the catalogue (for Lina, Loubna, Imane):

| Question | Profile field | Catalogue column(s) |
|---|---|---|
| Q1 age · Q2 concern · Q3 blemishes · Q4 special cases | `is_adult`, `concern`, `blemish_level`, `referral_flags` | none (referral rule) |
| Q5 skin type | `skin_type` | `is_rich`, `is_light` |
| Q6 reactivity | `reactivity` | `has_parfum`, `has_essential_oil` |
| Q7 avoid | `avoid` | `has_parfum`, `has_fragrance_allergen`, `has_essential_oil` |
| Concern (acne-prone) | — | `has_acne_active`, `is_anti_ageing` |

In [ ]:
contract = ["is_rich", "is_light", "has_parfum", "has_fragrance_allergen", "has_essential_oil", "has_acne_active", "is_anti_ageing"]
print("Columns missing in the catalogue:", [column for column in contract if column not in moist.columns])

In [ ]:
def profile_outcome(profile):
    """'out of scope', 'referral' or 'eligible', in the order of the questionnaire."""
    if profile["is_adult"] == False:
        return "referral"                  # Meet 26/10: under 18 = referral
    elif profile["concern"] != "acne_prone":
        return "out of scope"              # not a medical referral: Swan only covers acne-prone skin
    elif profile["blemish_level"] >= BLEMISH_REFERRAL_LEVEL:
        return "referral"
    elif len(profile["referral_flags"]) > 0:
        return "referral"
    else:
        return "eligible"

def eligible_products(profile, catalogue):
    """In-scope products left after the 'avoid' answers."""
    selected = catalogue[catalogue["in_scope"]]
    avoid = profile.get("avoid", [])        # Q7 is optional: default = nothing to avoid
    if "fragrance" in avoid:
        selected = selected[~selected["has_parfum"]]
        if AVOID_FRAGRANCE_INCLUDES_ALLERGENS:
            selected = selected[~selected["has_fragrance_allergen"]]
    if "essential_oils" in avoid:
        selected = selected[~selected["has_essential_oil"]]
    return selected

def simple_score(profile, selected):
    """PLACEHOLDER score — Loubna's score will replace it (T15). Each point has a reason."""
    score = selected["has_acne_active"].astype(int) - selected["is_anti_ageing"].astype(int)
    if profile["reactivity"] == True:                     # fragrance ranked lower, not removed (removal = Q7)
        score = score - selected["has_parfum"].astype(int) - selected["has_essential_oil"].astype(int)
    if profile["skin_type"] in LIGHT_SKIN_TYPES:          # light preferred, rich ranked lower (not removed)
        score = score + selected["is_light"].astype(int) - selected["is_rich"].astype(int)
    elif profile["skin_type"] == "dry":
        score = score + selected["is_rich"].astype(int)
    return score

def recommend(profile, catalogue):
    """Answers -> 'out of scope', 'referral', 'not enough products', or the top moisturizers."""
    outcome = profile_outcome(profile)
    if outcome != "eligible":
        return outcome
    selected = eligible_products(profile, catalogue).copy()
    selected["score"] = simple_score(profile, selected)
    selected = selected.sort_values(["score", "coverage"], ascending=False)
    selected = selected.drop_duplicates("name_brand")      # the same product never twice in a top 3
    if len(selected) < NB_ROUTINE_PRODUCT:
        return "not enough products"
    return selected.head(NB_ROUTINE_PRODUCT)

## Test some testing profiles

Avoid fragrance (Q7): removes products with `parfum` only
Reactive skin (Q6 = yes): fragrance and essential oils ranked lower, not removed
Oily, combination, normal, not sure: light texture preferred, rich ranked lower
Dry: rich texture preferred

In [ ]:
# Recommendation profiles: every skin type x reactivity x (nothing / fragrance) to avoid,
profiles = []
number = 1
for skin_type in QUESTIONNAIRE["skin_type"]:
    for reactivity in [True, False]:
        for avoid in [[], ["fragrance"]]:
            profiles.append({"profile_id": "P" + str(number).zfill(2), "kind": "recommendation",
                             "is_adult": True, "concern": "acne_prone", "blemish_level": number % 3 + 1,
                             "referral_flags": [], "skin_type": skin_type, "reactivity": reactivity,
                             "avoid": avoid, "expected": "products"})
            number = number + 1

# The 2 other 'avoid' answers, once each
for avoid in [["essential_oils"], ["fragrance", "essential_oils"]]:
    profiles.append({"profile_id": "P" + str(number).zfill(2), "kind": "recommendation",
                     "is_adult": True, "concern": "acne_prone", "blemish_level": 2,
                     "referral_flags": [], "skin_type": "combination", "reactivity": True,
                     "avoid": avoid, "expected": "products"})
    number = number + 1

# Edge cases: one per referral rule, + 2 reasons at once
edge_cases = [
    ("under 18",                       {"is_adult": False}),
    ("other concern",                  {"concern": "other", "expected": "out of scope"}),
    ("blemishes = 4",                  {"blemish_level": 4}),
    ("blemishes = 5",                  {"blemish_level": 5}),
    ("prescribed treatment",           {"referral_flags": ["prescribed_treatment"]}),
    ("unusual lesion",                 {"referral_flags": ["unusual_lesion"]}),
    ("pregnant",                       {"referral_flags": ["pregnant_or_breastfeeding"]}),
    ("pregnant + prescribed treatment", {"referral_flags": ["pregnant_or_breastfeeding", "prescribed_treatment"]}),
]
for name, change in edge_cases:
    profile = {"profile_id": "P" + str(number).zfill(2), "kind": "edge case: " + name,
               "is_adult": True, "concern": "acne_prone", "blemish_level": 2, "referral_flags": [],
               "skin_type": "oily", "reactivity": False, "avoid": [], "expected": "referral"}
    profile.update(change)
    profiles.append(profile)
    number = number + 1

profiles.append({"profile_id": "P" + str(number).zfill(2), "kind": "edge case: blemishes = 3 (just below)",
                 "is_adult": True, "concern": "acne_prone", "blemish_level": 3, "referral_flags": [],
                 "skin_type": "combination", "reactivity": True, "avoid": [], "expected": "products"})

def expected_traits(profile):
    """What a good product should respect for this profile, from the decisions above."""
    if profile["expected"] != "products":
        return "no products"
    traits = []
    if "fragrance" in profile["avoid"]:
        traits.append("no parfum")
    if "essential_oils" in profile["avoid"]:
        traits.append("no essential oil")
    if profile["reactivity"] == True:
        traits.append("fragrance-free first")
    if profile["skin_type"] in LIGHT_SKIN_TYPES:
        traits.append("not rich")
    elif profile["skin_type"] == "dry":
        traits.append("rich")
    return ", ".join(traits)

for profile in profiles:
    profile["expected_traits"] = expected_traits(profile)

profiles_table = pd.DataFrame(profiles)
print("Profiles:", len(profiles_table))
print(profiles_table["expected"].value_counts())
print(pd.crosstab(profiles_table["reactivity"], profiles_table["avoid"].astype(str)))   # check: reactivity and avoid are crossed
profiles_table

In [ ]:
for profile in profiles:
    answers = {field: profile[field] for field in QUESTIONNAIRE}
    assert check_profile(answers) == [], profile["profile_id"]

for field in ["skin_type", "reactivity", "blemish_level"]:
    print(field, ":", sorted(profiles_table[field].astype(str).unique()))
print("avoid :", profiles_table["avoid"].astype(str).unique())

In [ ]:
result_rows = []
for profile in profiles:
    result = recommend(profile, moist)
    if type(result) == str:
        result_rows.append({"profile_id": profile["profile_id"], "expected": profile["expected"], "outcome": result})
    else:
        rank = 1
        for _, product in result.iterrows():           # one loop turn per recommended product (row)
            respects_avoid = True
            if "fragrance" in profile["avoid"] and product["has_parfum"]:
                respects_avoid = False
            if "essential_oils" in profile["avoid"] and product["has_essential_oil"]:
                respects_avoid = False
            if profile["skin_type"] == "dry":
                texture_ok = product["is_rich"]
            else:
                texture_ok = not product["is_rich"]
            result_rows.append({"profile_id": profile["profile_id"], "expected": profile["expected"], "outcome": "products",
                                "rank": rank, "code": product["code"], "product_name": product["product_name"],
                                "score": product["score"], "R2_respects_avoid": respects_avoid,
                                "expected_traits": profile["expected_traits"], "texture_ok": texture_ok,
                                "R4_face_moisturizer": "", "R5_texture_fits": "", "comment": ""})
            rank = rank + 1

results = pd.DataFrame(result_rows)
results.head(10)

In [ ]:
outcome_per_profile = results.groupby("profile_id")[["expected", "outcome"]].first()
print("R1 expected outcome     :", (outcome_per_profile["expected"] == outcome_per_profile["outcome"]).sum(), "/", len(outcome_per_profile))

products = results[results["outcome"] == "products"]
print("R2 respects 'avoid'     :", products["R2_respects_avoid"].sum(), "/", len(products))
print("R3 no duplicate in a top:", (products.groupby("profile_id")["code"].nunique() == NB_ROUTINE_PRODUCT).sum(), "/", products["profile_id"].nunique())
print("Texture fits (automatic):", products["texture_ok"].sum(), "/", len(products), "  (rule: rich for dry skin, not rich otherwise)")

print()
print("Profiles with a wrong outcome:")
outcome_per_profile[outcome_per_profile["expected"] != outcome_per_profile["outcome"]]

In [ ]:
results.to_csv("../raw_data/swan_test_results_v2_final_catalogue.csv", index=False)
print("Saved: swan_test_results_v2_final_catalogue.csv —", len(products), "products to judge")

## Negative controls

Products that must never be in a top 3: not sold in France, or broken ingredient list.

In [ ]:
controls = list(moist[~moist["in_scope"]]["code"])
shown = list(products["code"])
wrong = [code for code in controls if code in shown]
print("Controls:", len(controls), "| shown in a top 3:", len(wrong))
assert wrong == []
print("Negative controls OK")

## Extra checks

In [ ]:
tiny = moist[moist["in_scope"]].head(2)       # a catalogue with only 2 products
base_profile = {"is_adult": True, "concern": "acne_prone", "blemish_level": 2, "referral_flags": [],
                "skin_type": "oily", "reactivity": False, "avoid": []}
assert recommend(base_profile, tiny) == "not enough products"

no_avoid = {k: v for k, v in base_profile.items() if k != "avoid"}   # Q7 not answered
assert type(recommend(no_avoid, moist)) != str                       # gets products, no crash
print("Extra checks OK")

## Tests on fictional products

6 invented products with known answers: we check the logic of `recommend()` without depending on the real data. These products are never recommended.

In [ ]:
fake = pd.DataFrame({
    "code":          ["F1", "F2", "F3", "F4", "F5", "F6"],
    "product_name":  ["Light gel", "Light fluid", "Rich cream", "Rich balm", "Parfum gel", "Oil cream"],
    "name_brand":    ["light gel", "light fluid", "rich cream", "rich balm", "parfum gel", "oil cream"],
    "in_scope":      [True, True, True, True, True, True],
    "has_parfum":    [False, False, False, False, True, False],
    "has_fragrance_allergen": [False, False, False, False, True, False],
    "has_essential_oil": [False, False, False, False, False, True],
    "has_acne_active": [False, False, False, False, False, False],
    "is_anti_ageing": [False, False, False, False, False, False],
    "is_light":      [True, True, False, False, True, False],
    "is_rich":       [False, False, True, True, False, True],
    "coverage":      [1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
})
base_profile = {"is_adult": True, "concern": "acne_prone", "blemish_level": 2, "referral_flags": [],
                "skin_type": "oily", "reactivity": False, "avoid": []}

def change(field, value):
    """A copy of base_profile with one answer changed."""
    profile = base_profile.copy()
    profile[field] = value
    return profile

# Routing
assert recommend(change("is_adult", False), fake) == "referral"
assert recommend(change("concern", "other"), fake) == "out of scope"
assert recommend(change("blemish_level", 4), fake) == "referral"

# Avoid answers are respected
result = recommend(change("avoid", ["fragrance"]), fake)
assert "F5" not in list(result["code"])
result = recommend(change("avoid", ["essential_oils"]), fake)
assert "F6" not in list(result["code"])

# Texture: light first for oily skin, rich first for dry skin
assert recommend(base_profile, fake).iloc[0]["is_light"] == True
assert recommend(change("skin_type", "dry"), fake).iloc[0]["is_rich"] == True

# The same product (same name + brand) never twice
twins = fake.copy()
twins.loc[1, "name_brand"] = "light gel"
result = recommend(base_profile, twins)
assert result["name_brand"].nunique() == len(result)

print("Tests on fictional OK")

In [ ]:
print("1 under 18      :", recommend(change("is_adult", False), fake))
print("2 other concern :", recommend(change("concern", "other"), fake))
print("3 blemishes 4   :", recommend(change("blemish_level", 4), fake))

result = recommend(change("avoid", ["fragrance"]), fake)
print("4 avoid parfum  :", result if type(result) == str else list(result["code"]))
result = recommend(change("avoid", ["essential_oils"]), fake)
print("5 avoid oils    :", result if type(result) == str else list(result["code"]))

result = recommend(base_profile, fake)
print("6 oily          :", result if type(result) == str else list(result["code"]))
result = recommend(change("skin_type", "dry"), fake)
print("7 dry           :", result if type(result) == str else list(result["code"]))

## Does each question change the result?

Start from one profile and change **one answer at a time**.

In [ ]:
base = {"is_adult": True, "concern": "acne_prone", "blemish_level": 2, "referral_flags": [],
        "skin_type": "oily", "reactivity": False, "avoid": []}

for question in ["skin_type", "reactivity", "avoid"]:
    print("=====", question)
    for answer in QUESTIONNAIRE[question]:
        profile = base.copy()
        profile[question] = answer
        result = recommend(profile, moist)
        if type(result) == str:
            print(answer, "→", result)
        else:
            print(answer, "→", list(result["product_name"]))

In [ ]:
tops = products.groupby("profile_id")["product_name"].apply(" + ".join)
print("Different top 3:", tops.nunique(), "out of", len(tops), "profiles with products")
print("Different products shown:", products["code"].nunique(), "out of", moist["in_scope"].sum(), "in scope")
print()
print("Most shown products:")
products["product_name"].value_counts().head(5)

# Tests with Loubna's model

Same 31 profiles and same checks, with `recommend()` from `models/recommendations.py` (Loubna's branch, copied locally, not modified).

Her statuses are not written like mine, so the adapter translates them:

| Loubna | Me |
|---|---|
| `ok` | top 3 products |
| `referral` | `referral` |
| `out_of_scope` | `out of scope` |
| `insufficient_results`, `no_result` | `not enough products` |

In [ ]:
import sys
sys.path.insert(0, "..")      # the Swan folder, where models/ is

from models.recommendations import recommend as loubna_model
from models.baseline import ranking_data     # her 50 products, with the ingredient columns her ranker needs

print("Loubna's products:", len(ranking_data), "| all in my in-scope list:", ranking_data["code"].isin(moist[moist["in_scope"]]["code"]).all())

In [ ]:
LOUBNA_STATUS = {"out_of_scope": "out of scope",
                 "insufficient_results": "not enough products",
                 "no_result": "not enough products"}

def loubna_recommend(profile, catalogue):
    """Call Loubna's model and give the answer in the same format as my recommend()."""
    answers = {field: profile[field] for field in QUESTIONNAIRE}
    result = loubna_model(answers, ranking_data)    # her ranker needs her own prepared file
    if result["status"] != "ok":
        return LOUBNA_STATUS.get(result["status"], result["status"])
    codes = [product["code"] for product in result["results"]]
    top = catalogue.set_index("code").loc[codes].reset_index()    # her order, with my signal columns
    top["score"] = [product["score"] for product in result["results"]]
    return top

print(loubna_recommend(base_profile, moist)[["code", "product_name", "score"]])

In [ ]:
result_rows = []
for profile in profiles:
    result = loubna_recommend(profile, moist)
    if type(result) == str:
        result_rows.append({"profile_id": profile["profile_id"], "expected": profile["expected"], "outcome": result})
    else:
        rank = 1
        for _, product in result.iterrows():
            respects_avoid = True
            if "fragrance" in profile["avoid"] and product["has_parfum"]:
                respects_avoid = False
            if "essential_oils" in profile["avoid"] and product["has_essential_oil"]:
                respects_avoid = False
            if profile["skin_type"] == "dry":
                texture_ok = product["is_rich"]
            else:
                texture_ok = not product["is_rich"]
            result_rows.append({"profile_id": profile["profile_id"], "expected": profile["expected"], "outcome": "products",
                                "rank": rank, "code": product["code"], "product_name": product["product_name"],
                                "score": product["score"], "R2_respects_avoid": respects_avoid,
                                "expected_traits": profile["expected_traits"], "texture_ok": texture_ok,
                                "R4_face_moisturizer": "", "R5_texture_fits": "", "comment": ""})
            rank = rank + 1

results_loubna = pd.DataFrame(result_rows)
results_loubna.to_csv("../raw_data/swan_test_results_v2_loubna_model.csv", index=False)
print("Saved: swan_test_results_v2_loubna_model.csv")

In [ ]:
def summary(results):
    """The same checks for both models."""
    per_profile = results.groupby("profile_id")[["expected", "outcome"]].first()
    shown = results[results["outcome"] == "products"]
    tops = shown.groupby("profile_id")["product_name"].apply(" + ".join)
    return {"R1 expected outcome": str((per_profile["expected"] == per_profile["outcome"]).sum()) + " / " + str(len(per_profile)),
            "R2 respects avoid": str(shown["R2_respects_avoid"].sum()) + " / " + str(len(shown)),
            "R3 no duplicate": str((shown.groupby("profile_id")["code"].nunique() == NB_ROUTINE_PRODUCT).sum()) + " / " + str(shown["profile_id"].nunique()),
            "Texture fits": str(shown["texture_ok"].sum()) + " / " + str(len(shown)),
            "Negative controls shown": shown["code"].isin(controls).sum(),
            "Different top 3": tops.nunique(),
            "Different products shown": shown["code"].nunique()}

pd.DataFrame({"My score": summary(results), "Loubna's model": summary(results_loubna)})

In [ ]:
# profiles where the two models don't give the same outcome
mine = results.groupby("profile_id")["outcome"].first()
hers = results_loubna.groupby("profile_id")["outcome"].first()
different = pd.DataFrame({"my score": mine, "Loubna's model": hers})
different[different["my score"] != different["Loubna's model"]]

## Look in detail: profile by profile

In [ ]:
# top 3 side by side for each profile
mine = results[results["outcome"] == "products"].groupby("profile_id")["product_name"].apply(" + ".join)
hers = results_loubna[results_loubna["outcome"] == "products"].groupby("profile_id")["product_name"].apply(" + ".join)

side_by_side = profiles_table.set_index("profile_id")[["skin_type", "reactivity", "avoid"]].copy()
side_by_side["my top 3"] = mine
side_by_side["Loubna's top 3"] = hers
side_by_side["same"] = side_by_side["my top 3"] == side_by_side["Loubna's top 3"]
side_by_side = side_by_side.dropna(subset=["my top 3"])
print("Same top 3:", side_by_side["same"].sum(), "/", len(side_by_side))

with pd.option_context("display.max_colwidth", None):
    display(side_by_side)

In [ ]:
# one profile in detail: change the id to look at another one
PROFILE_ID = "P13"      # a dry skin profile

profile = [p for p in profiles if p["profile_id"] == PROFILE_ID][0]
print({field: profile[field] for field in QUESTIONNAIRE})
print("Expected traits:", profile["expected_traits"])

signals_to_show = moist[["code", "has_parfum", "is_rich", "is_light", "has_acne_active", "is_anti_ageing"]]

print("\nMy score")
display(results[results["profile_id"] == PROFILE_ID][["code", "product_name", "score", "texture_ok"]].merge(signals_to_show, on="code"))
print("Loubna's model")
display(results_loubna[results_loubna["profile_id"] == PROFILE_ID][["code", "product_name", "score", "texture_ok"]].merge(signals_to_show, on="code"))

In [ ]:
# why Loubna's model chose these products (her 'reason' column)
answers = {field: profile[field] for field in QUESTIONNAIRE}
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(loubna_model(answers, ranking_data)["results"])[["product_name", "score", "reason"]])

In [ ]:
# products with a texture that does not fit the skin type
texture_problems = results_loubna[(results_loubna["outcome"] == "products") & (results_loubna["texture_ok"] == False)]
pd.crosstab(texture_problems["product_name"], profiles_table.set_index("profile_id").loc[texture_problems["profile_id"], "skin_type"].values)

## Experiment: Loubna's score + skin type and reactivity

Test only: Loubna's files are **not** changed. Her score (moisturising +1, acne active +1) + the questionnaire decisions of 01/11:
- oily, combination, normal, not sure: light +1, rich -1
- dry: rich +1
- reactive skin: parfum -1, essential oil -1

If it works, I show the results to Loubna and she decides if she adds it to her model.

In [ ]:
from models.recommendations import check_scope, filter_avoid

def loubna_plus(profile, catalogue):
    """Loubna's model + skin type and reactivity (experiment)."""
    answers = {field: profile[field] for field in QUESTIONNAIRE}
    status = check_scope(answers)
    if status != "eligible":
        return LOUBNA_STATUS.get(status, status)

    selected = filter_avoid(ranking_data, answers["avoid"]).copy()
    selected = selected.drop(columns=["is_rich", "is_light"], errors="ignore")
    selected = selected.merge(moist[["code", "is_rich", "is_light"]], on="code")    # my texture rule

    score = selected["moisturising_flag"].astype(int) + selected["has_acne_active"].astype(int)     # Loubna's score
    if answers["skin_type"] in LIGHT_SKIN_TYPES:
        score = score + selected["is_light"].astype(int) - selected["is_rich"].astype(int)
    elif answers["skin_type"] == "dry":
        score = score + selected["is_rich"].astype(int)
    if answers["reactivity"] == True:
        score = score - selected["has_parfum"].astype(int) - selected["has_essential_oil"].astype(int)
    selected["score"] = score

    selected = selected.sort_values(["score", "code"], ascending=[False, True])    # same tie rule as Loubna
    if len(selected) < NB_ROUTINE_PRODUCT:
        return "not enough products"
    codes = list(selected["code"].head(NB_ROUTINE_PRODUCT))
    top = catalogue.set_index("code").loc[codes].reset_index()
    top["score"] = list(selected["score"].head(NB_ROUTINE_PRODUCT))
    return top

In [ ]:
result_rows = []
for profile in profiles:
    result = loubna_plus(profile, moist)
    if type(result) == str:
        result_rows.append({"profile_id": profile["profile_id"], "expected": profile["expected"], "outcome": result})
    else:
        rank = 1
        for _, product in result.iterrows():
            respects_avoid = True
            if "fragrance" in profile["avoid"] and product["has_parfum"]:
                respects_avoid = False
            if "essential_oils" in profile["avoid"] and product["has_essential_oil"]:
                respects_avoid = False
            if profile["skin_type"] == "dry":
                texture_ok = product["is_rich"]
            else:
                texture_ok = not product["is_rich"]
            result_rows.append({"profile_id": profile["profile_id"], "expected": profile["expected"], "outcome": "products",
                                "rank": rank, "code": product["code"], "product_name": product["product_name"],
                                "score": product["score"], "R2_respects_avoid": respects_avoid,
                                "expected_traits": profile["expected_traits"], "texture_ok": texture_ok})
            rank = rank + 1

results_plus = pd.DataFrame(result_rows)

pd.DataFrame({"My score": summary(results),
              "Loubna's model": summary(results_loubna),
              "Loubna + skin type + reactivity": summary(results_plus)})

In [ ]:
# P13 again (dry, reactive)
results_plus[results_plus["profile_id"] == "P13"][["code", "product_name", "score", "texture_ok"]].merge(signals_to_show, on="code")